# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
333,29,20,"(0.5511096795536096, 0.8938255112494718)","(-0.084, 0.577, -0.012, -0.775)",0,1.0,"(-0.072, 0.384, -0.027, -0.534)",1.0,1.0,"(-0.064, 0.577, -0.038, -0.785)",...,-0.012,-0.775,-0.072,0.384,-0.027,-0.534,-0.064,0.577,-0.038,-0.785
112,3,8,"(0.3784842081013151, 0.1630291663537095)","(-0.028, -0.183, 0.05, 1.118)",1,1.0,"(-0.032, 0.048, 0.072, 0.054)",0.0,1.0,"(-0.031, -0.189, 0.073, 1.264)",...,0.050,1.118,-0.032,0.048,0.072,0.054,-0.031,-0.189,0.073,1.264
16,1,2,"(0.4504695427312523, 0.7386594270712393)","(-0.024, -0.214, 0.031, 0.234)",0,1.0,"(-0.029, -0.409, 0.035, 0.509)",0.0,1.0,"(-0.037, -0.603, 0.045, 0.786)",...,0.031,0.234,-0.029,-0.409,0.035,0.509,-0.037,-0.603,0.045,0.786
15,0,2,"(0.4504695427312523, 0.7386594270712393)","(-0.024, -0.02, 0.031, -0.041)",0,1.0,"(-0.024, -0.214, 0.031, 0.234)",0.0,1.0,"(-0.029, -0.409, 0.035, 0.509)",...,0.031,-0.041,-0.024,-0.214,0.031,0.234,-0.029,-0.409,0.035,0.509
219,0,15,"(0.0385460830243375, 0.3725217881160826)","(0.019, 0.032, -0.016, -0.046)",0,1.0,"(0.02, -0.167, -0.016, 0.324)",1.0,1.0,"(0.017, 0.032, -0.01, -0.058)",...,-0.016,-0.046,0.020,-0.167,-0.016,0.324,0.017,0.032,-0.010,-0.058


# Predict 

In [4]:
def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        ds = get_data_expanded(weights, expansions)
        
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = ds[f'estimated_s{d}']
    weights['step_A'] = weights.apply(lambda row: [[row[f'estimated_s{d}_model{m}'] for m,_ in enumerate(models)] for d in range(4)], axis=1)
    weights['step_b'] = weights.apply(lambda row: [row[f's__{d}'] for d in range(4)], axis=1)

    weights['concat_A'] = weights.apply(lambda row: np.array(weights.loc[(weights['episode']==row['episode'])&(weights['step']<=row['step'])].step_A.to_list()), axis=1)
    weights['concat_b'] = weights.apply(lambda row: np.array(weights.loc[(weights['episode']==row['episode'])&(weights['step']<=row['step'])].step_b.to_list()), axis=1)
    
    weights['A'] = weights.apply(lambda row: row.concat_A.reshape((row.concat_A.shape[0]*row.concat_A.shape[1], row.concat_A.shape[2])), axis=1)
    weights['b'] = weights.apply(lambda row: row.concat_b.reshape((row.concat_b.shape[0]*row.concat_b.shape[1],)), axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0], axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights


In [5]:

def predict_with_params(prediction_dataset, weights):
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']

    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    ds = get_data_expanded(prediction_dataset, expansions)
    
    for d in range(4):
        ds[f'estimated_weighted_s{d}'] = ds.apply(
            lambda row: np.sum([row[f'estimated_s{d}_model{m}']* row[f'weight_{m}'] for m, _ in enumerate(models)])
            , axis=1
        )
    
    ds = get_data_compacted(ds, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
     
    prediction_dataset['estimated_r'] = ds['estimated_r_model_0']
    prediction_dataset['estimated_s'] = ds['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [6]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [7]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        's__', 'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ] + [f'weight_{m}' for m, _ in enumerate(models)] + [f'estimated_s_model_{m}' for m, _ in enumerate(models)]

    return final_predictions[cols]

In [8]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

,s__,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,...,weight_0,weight_1,weight_2,weight_3,weight_4,estimated_s_model_0,estimated_s_model_1,estimated_s_model_2,estimated_s_model_3,estimated_s_model_4
333,"(-0.064, 0.577, -0.038, -0.785)","(-0.06275088610316501, 0.5778420717097887, -0....",0.022129,0.503384,0.001249,0.000842,0.001052,0.018986,0.547254,0.526264,...,0.601886,0.014562,0.382270,-0.000669,0.001062,"(-0.062, 0.578, -0.04, -0.735)","(-0.064, 0.575, -0.039, -0.684)","(-0.064, 0.579, -0.038, -0.816)","(-0.069, 0.589, -0.036, -0.996)","(-0.078, 0.601, 0.088, -2.261)"
112,"(-0.031, -0.189, 0.073, 1.264)","(-0.027710779565672194, -0.17972202548967475, ...",0.178797,0.515014,0.003289,0.009278,0.013932,0.152298,0.549408,0.528338,...,0.127955,-0.105441,-0.446983,0.896549,0.484306,"(-0.031, -0.144, 0.074, 0.284)","(-0.031, -0.14, 0.073, 0.185)","(-0.031, -0.149, 0.074, 0.425)","(-0.031, -0.161, 0.073, 0.659)","(-0.027, -0.203, 0.109, 1.433)"
16,"(-0.037, -0.603, 0.045, 0.786)","(-0.04043668103132224, -0.6086915197324851, 0....",0.015115,0.504548,0.003437,0.005692,0.002071,0.003915,0.549564,0.527456,...,0.067273,-0.218461,1.686328,-0.530249,0.004714,"(-0.038, -0.601, 0.045, 0.735)","(-0.035, -0.596, 0.046, 0.683)","(-0.038, -0.606, 0.044, 0.838)","(-0.035, -0.616, 0.045, 1.029)","(-0.001, -0.675, -0.082, 3.04)"
15,"(-0.029, -0.409, 0.035, 0.509)","(-0.026744279054122574, -0.40916722408921624, ...",0.046205,0.505512,0.002256,0.000167,0.003467,0.040315,0.548316,0.526098,...,0.176964,0.497212,0.221728,0.039733,0.065131,"(-0.028, -0.406, 0.036, 0.451)","(-0.028, -0.402, 0.036, 0.37)","(-0.028, -0.41, 0.035, 0.569)","(-0.027, -0.422, 0.035, 0.772)","(-0.009, -0.457, -0.029, 1.976)"
219,"(0.017, 0.032, -0.01, -0.058)","(0.018154778637416432, 0.03329315100035459, -0...",0.039846,0.504226,0.001155,0.001293,0.001861,0.035537,0.547154,0.526375,...,-0.011557,0.629447,-0.430039,0.720516,0.079681,"(0.017, 0.026, -0.011, 0.091)","(0.017, 0.023, -0.01, 0.161)","(0.017, 0.032, -0.01, -0.012)","(0.018, 0.038, -0.01, -0.195)","(0.025, 0.069, -0.035, -0.734)"


In [9]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,weight_0,weight_1,weight_2,weight_3,weight_4
count,2127.000000,2127.000000,2.127000e+03,2127.000000,2.127000e+03,2.127000e+03,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.081597,0.507555,3.741037e-03,0.007116,4.587030e-03,6.615228e-02,0.549885,0.527806,0.512199,0.440331,0.408950,0.079346,0.226785,0.224502,0.054324
std,0.512780,0.030678,1.745024e-02,0.046002,2.979140e-02,4.505131e-01,0.018427,0.011308,0.071442,0.038548,0.602911,0.683830,0.937855,0.674456,0.258947
min,0.000301,0.502024,5.795782e-07,0.000003,5.088600e-09,2.534313e-07,0.545935,0.526058,0.501199,0.434671,-4.924445,-17.171353,-16.175301,-7.009247,-0.357158
25%,0.005893,0.502647,4.060967e-04,0.000651,4.471519e-04,2.945020e-03,0.546363,0.526217,0.502271,0.434923,0.039518,-0.104039,-0.175450,-0.041257,-0.001873
50%,0.012022,0.503204,9.584189e-04,0.001512,1.033059e-03,7.841694e-03,0.546947,0.526429,0.503676,0.435342,0.392688,0.054755,0.223718,0.017218,0.000184
75%,0.030502,0.504406,1.968328e-03,0.003694,2.135559e-03,2.095877e-02,0.548013,0.526965,0.506320,0.436464,0.737524,0.316167,0.660353,0.259845,0.004789
max,10.900556,1.408025,4.177414e-01,1.887649,1.023576e+00,1.036722e+01,0.987055,0.990081,2.955818,1.321744,8.499554,6.646667,14.684618,9.614709,3.788297
